In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

#### Create `openalex.funders.openalex_funders_snapshot` in same format as API

In [ ]:
df_transformed = (
    spark.read.table("openalex.funders.funders_api")
    # Convert BIGINT id to full URL
    .withColumn("id", F.concat(F.lit("https://openalex.org/F"), F.col("id")))
    # Coalesce null arrays to empty arrays
    .withColumn("alternate_titles", F.coalesce(F.col("alternate_titles"), F.array()))
    .withColumn("roles", F.coalesce(F.col("roles"), F.array()))
    .withColumn("counts_by_year", F.coalesce(F.col("counts_by_year"), F.array()))
)

df_transformed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("openalex.funders.openalex_funders_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/funders/`.

In [0]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.funders.openalex_funders_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "funders", salt=False)